# BassSpecMatchPRO — NAM Colab
Execute a célula abaixo. Ela inicia o worker temporário; o timbre vem somente da referência e o app envia o par **input.wav + output.wav** para treinamento NAM real.


In [ ]:
# BassSpecMatchPRO public worker — Colab/GPU, input-output contract v2
import os, sys, subprocess, secrets, json, time, threading, zipfile, shutil, re, urllib.request, base64, wave, socket
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'flask', 'neural-amp-modeler==0.13.0'])
from flask import Flask, request, jsonify
CLOUDFLARED='/content/cloudflared'
if not (os.path.isfile(CLOUDFLARED) and os.access(CLOUDFLARED, os.X_OK)):
    tmp=CLOUDFLARED+'.download-'+secrets.token_hex(4)
    try:
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', tmp)
        os.chmod(tmp, 0o755)
        os.replace(tmp, CLOUDFLARED)
    finally:
        if os.path.exists(tmp): os.remove(tmp)
ROOT=Path('/content/bassspec_nam_worker'); ROOT.mkdir(parents=True, exist_ok=True)
TOKEN=secrets.token_urlsafe(32)
state={'state':'ready','phase':'waiting','epoch':0,'epochs':0,'error':'','command':''}
app=Flask(__name__)
def auth(): return request.headers.get('Authorization','') == 'Bearer '+TOKEN

def validate_job(work):
    manifest=json.loads((work/'manifest.json').read_text())
    if manifest.get('format')!='bassspec-nam-colab-job-v2' or manifest.get('schema_version')!=2:
        raise ValueError('job/schema Colab v2 inválido')
    if manifest.get('training_mode')!='input-output':
        raise ValueError('training_mode deve ser input-output')
    if manifest.get('input_file')!='input.wav' or manifest.get('output_file')!='output.wav':
        raise ValueError('manifest input/output inválido')
    if manifest.get('reference_timbre_only') is not True or manifest.get('contains_source_audio') is not False:
        raise ValueError('proveniência do job inválida')
    meta=[]
    for name in ('input.wav','output.wav'):
        path=work/name
        if not path.is_file():
            raise FileNotFoundError(name+' ausente; fallback --reference proibido')
        with wave.open(str(path),'rb') as w:
            info=(w.getframerate(),w.getnchannels(),w.getnframes())
            if info[0]!=manifest.get('sample_rate') or info[1]!=manifest.get('channels') or info[2]<=0:
                raise ValueError(name+' incompatível com manifest')
            frames=w.readframes(info[2])
            if not frames or all(b==0 for b in frames):
                raise ValueError(name+' silencioso')
            meta.append(info)
    if meta[0]!=meta[1]:
        raise ValueError('input/output possuem formato ou comprimento incompatível')
    if (work/'input.wav').read_bytes()==(work/'output.wav').read_bytes():
        raise ValueError('output.wav não difere de input.wav')
    return manifest

@app.get('/health')
def health(): return jsonify({'ok':True}) if auth() else ('Unauthorized',401)

@app.post('/job')
def job():
    if not auth(): return ('Unauthorized',401)
    if state['state']=='training': return jsonify({'ok':True,'state':'running'})
    try:
        payload=request.get_json(force=True)
        epochs=int(payload.get('epochs',25)); epochs=100 if epochs>=100 else 25
        shutil.rmtree(ROOT/'job',ignore_errors=True); (ROOT/'job').mkdir(parents=True)
        z=ROOT/'job.zip'; z.write_bytes(base64.b64decode(payload['job_b64']))
        with zipfile.ZipFile(z) as f: f.extractall(ROOT/'job')
        validate_job(ROOT/'job')
    except Exception as e:
        state.update(state='error',phase='rejected',error=str(e),command='')
        return jsonify({'ok':False,'error':str(e)}),400
    state.update(state='received',phase='queued',epoch=0,epochs=epochs,error='',command='')
    return jsonify({'ok':True,'state':'running','epochs':epochs})

@app.get('/status')
def status(): return jsonify(state) if auth() else ('Unauthorized',401)

@app.get('/result')
def result():
    if not auth(): return ('Unauthorized',401)
    p=ROOT/'result'/'trained.nam'
    if state['state']!='done' or not p.exists(): return jsonify({'error':'not ready'}),409
    return jsonify({'ok':True,'nam_b64':base64.b64encode(p.read_bytes()).decode()})

def train_loop():
    while True:
        if state['state']=='received':
            try:
                work=ROOT/'job'; validate_job(work)
                epochs=int(state.get('epochs',25)); out=ROOT/'result'; shutil.rmtree(out,ignore_errors=True); out.mkdir()
                cmd=[sys.executable,str(work/'train_nam_official.py'),'--input',str(work/'input.wav'),'--output',str(work/'output.wav'),'--outdir',str(out),'--template',str(work/'model.nam'),'--epochs',str(epochs),'--ignore-checks']
                state.update(state='training',phase='training',command=' '.join(cmd))
                proc=subprocess.Popen(cmd,cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
                progress=out/'training-progress.json'
                while proc.poll() is None:
                    if progress.exists():
                        try:
                            d=json.loads(progress.read_text()); state.update(phase=d.get('phase','training'),epoch=int(d.get('epoch',0)),epochs=int(d.get('epochs',epochs)))
                        except Exception: pass
                    time.sleep(1)
                log=proc.stdout.read() if proc.stdout else ''
                if proc.returncode: raise RuntimeError(log[-4000:] or ('trainer failed: '+str(proc.returncode)))
                model=out/'trained.nam'; json.loads(model.read_text())
                state.update(state='done',phase='complete',epoch=epochs,epochs=epochs)
            except Exception as e:
                state.update(state='error',phase='error',error=str(e))
        time.sleep(.5)
threading.Thread(target=train_loop,daemon=True).start()
_port_probe=socket.socket(); _port_probe.bind(('127.0.0.1',0)); PORT=_port_probe.getsockname()[1]; _port_probe.close()
threading.Thread(target=lambda: app.run(host='127.0.0.1',port=PORT,use_reloader=False),daemon=True).start()
time.sleep(1)
tunnel=subprocess.Popen([CLOUDFLARED,'tunnel','--url','http://127.0.0.1:'+str(PORT),'--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
public=None; deadline=time.time()+45
while time.time()<deadline:
    line=tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None: break
        time.sleep(.1); continue
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',line)
    if m: public=m.group(0); break
if not public:
    tunnel.terminate(); raise RuntimeError('Cloudflare Quick Tunnel não iniciou; execute a célula novamente.')
print('BASSSPEC_COLAB='+public+'|'+TOKEN, flush=True)
print('Worker v2 pronto: o app enviará input.wav + output.wav e o treino usará --input/--output.')
